# Similarity and Record Linkage

## Objective
Compare Bloom filters and identify matches.

In [1]:
# =================== BOOTSTRAP CELL ===================
# Standard setup for all notebooks
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from src.config.variables import COVARIATES

# ========================================================
# Optional for warnings and nicer plots
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

import sys
from pathlib import Path

# ========================================================
# 1️⃣ Ensure project root is in Python path
# Adjust this if your notebooks are nested deeper
PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
# 2️⃣ Import helper to load paths
from src.utils.helpers import load_paths

# ========================================================
# 3️⃣ Load paths from config.yaml (works regardless of notebook location)
paths = load_paths()

# ========================================================
# 4️⃣ Optionally, print paths to confirm
for key, value in paths.items():
    print(f"{key}: {value}")

# ========================================================
# 5️⃣ Now you can use these paths in your notebook:
# Example:
SYNTHETIC_DATA_DIR = paths['SYNTHETIC_DATA_DIR']
PROCESSED_DATA_DIR = paths['PROCESSED_DATA_DIR']
# FIG_DIR = paths['FIG_DIR']

# ========================================================

BASE_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters
DATA_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data
OUT_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\model_output
FIG_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\visualization
MODEL_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\model_output\statsmodels
NOTEBOOKS_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\pprl_notebooks
NOTEBOOKS_EXECUTED_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\notebooks_executed
RAW_DATA_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data\raw
PROCESSED_DATA_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data

In [2]:
import pandas as pd

df_A = pd.read_pickle(PROCESSED_DATA_DIR / "df_A_blocked.pkl")
df_B = pd.read_pickle(PROCESSED_DATA_DIR / "df_B_blocked.pkl")

### Dice similarity

In [3]:
# Calculate similarity score
def dice_similarity(bf1, bf2):
    intersection = (bf1 & bf2).count()
    return 2 * intersection / (bf1.count() + bf2.count() + 1e-10) # 0.0000000001 to prevent division by zero

### PRE-EXTRACT DATA (CRITICAL OPTIMIZATION)

In [4]:
bloom_A = df_A["bloom"].to_dict()
bloom_B = df_B["bloom"].to_dict()
id_A_map = df_A["id"].to_dict()
id_B_map = df_B["id"].to_dict()

### Generating pairs

In [5]:
def generate_pairs(df_A, df_B, block):
    pairs = set()
    
    for val in df_A[block].dropna().unique(): # Get unique values from the blocking column.
        subA_idx = df_A[df_A[block] == val].index
        subB_idx = df_B[df_B[block] == val].index

        for i in subA_idx:
            for j in subB_idx:
                pairs.add((i, j))
    return pairs

print("\nGenerating candidate pairs using blocking...")

pairs = set()
for block_col in ["block1", "block2", "block3"]:# Trying different blocking columns (we dont want to miss potential matches)
    block_pairs = generate_pairs(df_A, df_B, block_col)
    print(f"{block_col}: {len(block_pairs)} pairs")
    pairs.update(block_pairs)

print(f"\nTotal unique candidate pairs: {len(pairs)}")

# matches = []
# for i, j in pairs: # Scoring and filtering matches.
#     sim = dice_similarity(df_A.loc[i,"bloom"], df_B.loc[j,"bloom"])
#     if sim >= 0.85: # If similarity is ≥ 85% consider it a match 
#         matches.append({
#             "id_A": df_A.loc[i,"id"],
#             "id_B": df_B.loc[j,"id"],
#             "sim": sim
#         })


Generating candidate pairs using blocking...


block1: 133266 pairs
block2: 192481 pairs
block3: 24403 pairs

Total unique candidate pairs: 321017


### Generating pairs - updated version

In [6]:
from joblib import Parallel, delayed



def compare_pair(i, j):
    # bf1 = df_A.loc[i, "bloom"]
    # bf2 = df_B.loc[j, "bloom"]
    # sim = dice_similarity(bf1, bf2)
    sim = dice_similarity(bloom_A[i], bloom_B[j])

    return (i, j, sim)


In [7]:
# Uses all CPUs
def parallel_compare(pairs, threshold=0.85):

    results = Parallel(
        n_jobs=-1,              # Use all CPUs
        batch_size=1000,        # Prevents memory overload 
        backend="loky"
        )(  
        delayed(compare_pair)(i, j)    # Each CPU runs compare_pair() independently
        for i, j in pairs
    )

    '''
    Execution flow: pairs → split → multiple CPUs → compute similarities → combine results
    '''

    print("Filtering matches...")

    # Filter matches
    matches = [
        {
            "id_A": id_A_map[i],
            "id_B": id_B_map[j],
            "similarity": sim
        }
        for i, j, sim in results 
        if sim >= threshold
    ]

    return matches

### LINKAGE

In [8]:
THRESHOLD = 0.85

matches = parallel_compare(pairs, threshold=THRESHOLD)
matches_df = pd.DataFrame(matches)

print(f"\nTotal matches found: {len(matches_df)}")

output_path = PROCESSED_DATA_DIR / "matches_optimized.csv"

matches_df.to_csv(output_path, index=False)

print(f"\nMatches saved to: {output_path}")

Filtering matches...

Total matches found: 751

Matches saved to: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data\processed\matches_optimized.csv


In [9]:
print("\nSample matches:")
print(matches_df.head())


Sample matches:
   id_A  id_B  similarity
0   417   417    0.877005
1   888   888    1.000000
2   451   377    0.862069
3   246   246    0.853801
4   966   966    0.896970
